In [ ]:
from ehda_normalization import prepare_training_data
from ehda_classifier import train
import sys
import os
from pathlib import Path

In [ ]:
# Add project root to Python path
project_root = Path(r"C:\Users\HV\Desktop\bruno_work\main")  # Goes up 3 levels to 'main/'
sys.path.insert(0, str(project_root))
from mapping.software.database import ElectrosprayDatabase # Import your DB class

# 0 - Init DB
BASE = Path(r"C:\Users\HV\Desktop\bruno_work\main\data")
db = ElectrosprayDatabase(str(BASE))


# 1. Load all data - Path to databse
df = db.load_training_dataframe()

In [7]:
df

,id,timestamp,solution_name,hv_position,target_voltage,actual_voltage,actual_current_ps,flow_rate,mean_na,deviation_na,...,band_power_low,band_power_mid,band_power_high,band_power_v_high,rf_spray_mode,xgb_spray_mode,image_classification,manual_classification,video_file,raw_data_file
0,1,2026-04-23T12:13:13.486060,EW82,nozzle,3000.0,3000.48,4.815040e-08,1.0,-9.475849,10.533339,...,49.701972,41.815113,5.052549,1.419719e-10,dripping (34%),dripping (81%),intermitent (100%),intermitent,2026-04-23_12-12-47_EW82.mp4,wf_2026-04-23_12-13-13_486060.npy
1,2,2026-04-23T12:13:18.174207,EW82,nozzle,3200.0,3200.38,5.413520e-08,1.0,-1.104915,23.670952,...,377.396431,113.566176,38.140785,9.807355e-10,multi_jet (29%),dripping (61%),dripping (96%),dripping,2026-04-23_12-12-47_EW82.mp4,wf_2026-04-23_12-13-18_174207.npy
2,3,2026-04-23T12:13:22.844423,EW82,nozzle,3400.0,3400.63,5.477000e-08,1.0,-15.125146,26.739436,...,606.451320,67.540646,16.430942,1.063364e-09,multi_jet (47%),intermitent (80%),dripping (98%),dripping,2026-04-23_12-12-47_EW82.mp4,wf_2026-04-23_12-13-22_844423.npy
3,4,2026-04-23T12:13:27.489614,EW82,nozzle,3600.0,3600.50,5.368180e-08,1.0,-10.035238,31.728207,...,962.085768,18.989270,5.531595,1.380294e-09,multi_jet (49%),intermitent (86%),intermitent (80%),intermitent,2026-04-23_12-12-47_EW82.mp4,wf_2026-04-23_12-13-27_489614.npy
4,5,2026-04-23T12:13:32.135037,EW82,nozzle,3800.0,3800.61,5.603950e-08,1.0,-11.275050,30.918859,...,811.633663,30.249796,5.303376,1.064484e-09,multi_jet (46%),intermitent (64%),dripping (100%),dripping,2026-04-23_12-12-47_EW82.mp4,wf_2026-04-23_12-13-32_135037.npy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3256,3936,2026-04-30T13:36:45.269662,EWG343,nozzle,8300.0,8300.53,1.052140e-06,3.0,693.477439,31.627970,...,48.591972,358.547564,13.395689,6.264630e-10,multi_jet (79%),multi_jet (100%),multi_jet (100%),multi_jet,2026-04-30_13-29-05_EWG343.mp4,wf_2026-04-30_13-36-45_269662.npy
3257,3937,2026-04-30T13:36:49.935982,EWG343,nozzle,8500.0,8500.37,8.779520e-07,3.0,635.750721,9.433226,...,17.192407,15.779860,4.859821,8.487790e-11,multi_jet (92%),multi_jet (100%),multi_jet (100%),multi_jet,2026-04-30_13-29-05_EWG343.mp4,wf_2026-04-30_13-36-49_935982.npy
3258,3938,2026-04-30T13:36:54.595290,EWG343,nozzle,8700.0,8700.53,9.750690e-07,3.0,613.034390,198.033030,...,21.244128,14.285095,5.487804,1.988626e-10,multi_jet (89%),multi_jet (100%),multi_jet (100%),multi_jet,2026-04-30_13-29-05_EWG343.mp4,wf_2026-04-30_13-36-54_595290.npy
3259,3939,2026-04-30T13:36:59.306381,EWG343,nozzle,8900.0,8900.08,8.691560e-07,3.0,324.573193,17.775430,...,33.571129,246.907734,15.435630,4.040023e-10,multi_jet (82%),multi_jet (99%),multi_jet (100%),multi_jet,2026-04-30_13-29-05_EWG343.mp4,wf_2026-04-30_13-36-59_306381.npy


In [ ]:
exclude_labels = ['EXCLUDE', 'unconclusive', 'undefined', 'N/A']

# 3. Clean the DataFrame
df_clean = df[~df['manual_classification'].isin(exclude_labels)].copy()

# 4. VALIDATION (NEW!)
print("Total samples before cleaning:", len(df))
print("Total samples after cleaning:", len(df_clean))
print("Remaining classes:\n", df_clean['manual_classification'].value_counts())


In [ ]:
df_clean


In [ ]:
if len(df_clean) == 0:
    raise ValueError("No samples left after cleaning!")

if df_clean['label'].nunique() < 2:
    raise ValueError("Need at least 2 classes!")